**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Performance Engineering & the Roofline Model

The unifying diagram behind every 'why is this slow' conversation in [GPU](../Intro_GPU/README.md), [Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb), and [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb): measure your machine's compute roof and memory roof, place your kernels on the chart, and *know* which wall you're hitting before touching a line of code.

## 1. Pre-requisites

[Intro to GPU Systems](../Intro_GPU/Intro_GPU.ipynb) (the CGMA idea), [Intro to OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) (caches exist).

In [1]:
import numpy as np
import time
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def bench(fn, *args, reps=7):
    ts = []
    for _ in range(reps):
        tic = time.perf_counter(); fn(*args); ts.append(time.perf_counter()-tic)
    return min(ts)                                  # min = least OS interference

---
### 🕐 Session 1 of 2 — *Measuring Your Machine's Roofs* (~40 min)
**Goal:** peak FLOP/s from matmul, peak GB/s from streaming — the two ceilings of all performance.
**Feeds into:** Session 2 (placing kernels on the roofline).

---

## 2. Two Ceilings

💡 **Intuition.** Every kernel is limited by one of two machine properties: how fast it can **compute** (FLOP/s, ceiling set by matmul-class code) or how fast it can **move data** (bytes/s, ceiling set by streaming). Which one binds is decided by the kernel's **arithmetic intensity** — FLOPs per byte touched — the same quantity [Intro_GPU](../Intro_GPU/Intro_GPU.ipynb) called CGMA. Below the machine's critical intensity, no cleverness in the arithmetic helps: you are paying for trucks, not workers.

In [2]:
# roof 1: compute (large matmul → BLAS at near-peak)
N = 2048
A = rng.standard_normal((N, N)).astype(np.float32)
B = rng.standard_normal((N, N)).astype(np.float32)
t_mm = bench(lambda: A @ B)
peak_flops = 2*N**3 / t_mm
print(f"compute roof:  {peak_flops/1e9:6.1f} GFLOP/s   (float32 matmul)")

# roof 2: memory bandwidth (pure streaming: y = x copy/scale of a cache-busting array)
M = 2**26                                              # 256 MB — far beyond cache
x = rng.standard_normal(M).astype(np.float32)
y = np.empty_like(x)
t_bw = bench(lambda: np.copyto(y, x))
peak_bw = 2*4*M / t_bw                                 # read + write, 4 bytes each
print(f"memory roof:   {peak_bw/1e9:6.1f} GB/s        (large-array copy)")
crit = peak_flops/peak_bw
print(f"critical intensity: {crit:.1f} FLOP/byte — below this, memory rules; above, compute rules")

compute roof:  1415.1 GFLOP/s   (float32 matmul)


memory roof:     41.7 GB/s        (large-array copy)
critical intensity: 34.0 FLOP/byte — below this, memory rules; above, compute rules


---
### 🕐 Session 2 of 2 — *Kernels on the Roofline* (~40 min)
**Goal:** place real operations on the chart; watch intensity, not effort, decide their fate.
**Builds on:** Session 1.

---

## 3. The Chart That Ends Arguments

In [3]:
# measure several kernels: achieved FLOP/s vs arithmetic intensity
kernels = {}

# saxpy: 2 FLOPs per 12 bytes → intensity 0.167 (hopelessly memory-bound)
a_s = np.float32(2.0)
kernels["saxpy"] = (2*M/ bench(lambda: a_s*x + y), 2/12)

# elementwise exp: ~1 'FLOP' per 8 bytes (in truth many flops inside exp — we count 1 op)
kernels["exp"] = (M/ bench(lambda: np.exp(x[:M//4])) /0.25, 1/8)

# small matmul (fits cache) vs large: same math, different effective intensity
for n_s, label in [(64, "matmul 64 (cache-warm)"), (2048, "matmul 2048")]:
    As = rng.standard_normal((n_s, n_s)).astype(np.float32)
    Bs = rng.standard_normal((n_s, n_s)).astype(np.float32)
    t = bench(lambda: As @ Bs, reps=15)
    kernels[label] = (2*n_s**3/t, n_s/6)          # intensity ≈ 2n³/(3·4n²) = n/6 FLOP/byte

ai = np.logspace(-1.2, 3, 200)
roof = np.minimum(peak_flops, ai*peak_bw)
plt.figure(figsize=(8, 3.4))
plt.loglog(ai, roof/1e9, "k", linewidth=1.5, label="the roofline")
for name, (flops, inten) in kernels.items():
    plt.plot(inten, flops/1e9, "o", markersize=8)
    plt.annotate(name, (inten, flops/1e9), textcoords="offset points", xytext=(6, 4), fontsize=7)
plt.axvline(crit, color="gray", linestyle=":", linewidth=0.8)
plt.xlabel("arithmetic intensity [FLOP/byte]"); plt.ylabel("achieved GFLOP/s")
plt.title("your machine's roofline, with real kernels placed on it")
plt.grid(True, which="both", alpha=0.3); plt.tight_layout(); plt.show()

for name, (flops, inten) in kernels.items():
    bound = "memory-bound" if inten < crit else "compute-bound"
    ceiling = min(peak_flops, inten*peak_bw)
    print(f"{name:22s}: {flops/1e9:7.1f} GFLOP/s = {flops/ceiling:4.0%} of its {bound} ceiling")

saxpy                 :     1.6 GFLOP/s =  23% of its memory-bound ceiling
exp                   :    11.9 GFLOP/s = 228% of its memory-bound ceiling
matmul 64 (cache-warm):   117.3 GFLOP/s =  26% of its memory-bound ceiling
matmul 2048           :  1458.9 GFLOP/s = 103% of its compute-bound ceiling


/tmp/ipykernel_319620/4157263946.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True, which="both", alpha=0.3); plt.tight_layout(); plt.show()


💡 **Intuition.** The diagnosis is now mechanical: a kernel far *below* its roof has implementation problems (fix the code); a kernel *on* a memory roof can only be helped by **raising its intensity** — fuse operations, tile for cache ([HW-Accelerated Computing's](../Intro_GPU/HW_Accelerated_Computing.ipynb) shared-memory story), or change algorithm. Optimizing a memory-bound kernel's arithmetic is polishing the truck's engine while it waits at the loading dock.

**The habit:** before optimizing anything, compute its intensity on a napkin and place it on this chart. Half of all optimization effort in the wild is spent on the wrong side of the critical intensity.

## 4. Conclusion

Two measured roofs, one intensity axis, every kernel placed — and the fix (better code vs more reuse vs different algorithm) read directly off the chart.

---
## Where next

- [HW-Accelerated Computing](../Intro_GPU/HW_Accelerated_Computing.ipynb) — raising intensity with shared memory.
- [Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) — rooflines at training scale.